# BorakBot — Base-model bake-off (Colab)

Picks the QLoRA base model by measuring the candidates instead of asserting one.
Produces the evidence behind `docs/Archive/model_selection.md`: a blinded Likert sheet and
an objective refusal table.

**Runtime → Change runtime type → T4 GPU** before running. On CPU this will not finish.

Colab rather than Kaggle because Kaggle gates notebook networking behind phone
verification, which this account cannot complete. Recorded as a deviation from the
Part A plan — see `docs/Archive/DESIGN.md`. Colab has networking on by default, so there is
no switch to set here.

Run the cells **in order**. `/content` is wiped on every disconnect, and Colab
disconnects more readily than Kaggle did, so results go to Drive in Cell 5.

    code     -> GitHub
    models   -> Hugging Face (re-downloaded after each disconnect)
    results  -> Google Drive

This notebook does NOT train anything. It only generates replies and counts
refusals; the human rating happens off-Colab in a spreadsheet.


## Cell 1 — Setup

Mounts Drive, installs the generation stack, clones the repo. Two to three minutes.
**Re-run this after every disconnect, and after every fix pushed to GitHub.**

`peft` is installed even though nothing is fine-tuned yet, because the same
`generate.py` is re-run at Step 4 with `--adapter`.

`/content/NLP-BorakBot` is Colab's local disk, **not** Google Drive. Drive is mounted
at `/content/drive/MyDrive/`. Renaming a Drive folder has no effect on the clone; the
only cell that touches Drive is Cell 5, which writes the results out.

Three things in this cell exist because each one has already gone wrong:

1. **`os.chdir('/content')` first.** A later cell leaves the working directory inside
   the clone. Deleting the directory you are standing in leaves the shell with no
   valid cwd, and every command after it dies with
   `getcwd: cannot access parent directories`.
2. **`shutil.rmtree` on a path constant, not `!rm -rf`.** A shell `rm -rf` splits an
   unquoted path on spaces: `!rm -rf /content/NLP Assignment` deletes `/content/NLP`
   *and* `Assignment`, removes nothing intended, and leaves the stale clone behind.
   The same slip pointed under `/content/drive` deletes real work.
3. **`check=True` plus an assert on the cloned file.** A failed clone otherwise prints
   an error, finishes green, and the next cells run old code — which is how a MaLLaM
   re-run reproduced the exact bug it was meant to fix.

If pip reports a version conflict with Colab's preinstalled `transformers`, use
**Runtime → Restart session** and re-run this cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q -U transformers peft bitsandbytes accelerate

import os, pathlib, shutil, subprocess

REPO = pathlib.Path('/content/NLP-BorakBot')
URL = 'https://github.com/yongvay/NLP-BorakBot.git'

# Step out of REPO before removing it. Deleting the directory the process is
# standing in leaves an invalid cwd, and everything after fails on getcwd.
os.chdir('/content')

# No shell, so no word-splitting on the path and nothing outside REPO can be hit.
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '-q', URL, str(REPO)], check=True, cwd='/content')

%cd /content/NLP-BorakBot
!git log --oneline -1

# Fail here rather than three cells later on a run that looks fine.
src = (REPO / 'eval' / 'generate.py').read_text(encoding='utf-8')
assert '--chat-format' in src, 'STALE CLONE: re-run this cell before going on'
print('clone ok: generate.py supports --chat-format')

import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## Cell 2 — Hugging Face token

**Required.** `meta-llama/Llama-3.2-3B-Instruct` is gated. Access has been granted
on the account, but the runtime still has to authenticate as that account.

Add it through the **key icon in the left sidebar** (Colab Secrets): name `HF_TOKEN`,
paste the value, and switch on *Notebook access*. Never paste the token into a cell —
notebook source gets committed, output gets shared, and a pasted token is a leaked
token.

MaLLaM is ungated and loads without this, so a failure here shows up later as one
candidate missing rather than as an error. Check the confirmation this prints.


In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    login(userdata.get('HF_TOKEN'))
    print('HF login ok')
except Exception as e:
    print('NO HF TOKEN - the gated Llama candidate will fail to download.')
    print('Left sidebar key icon -> add HF_TOKEN -> enable Notebook access.')
    print(' ', e)


## Cell 3 — Generate

Each candidate answers the same 20 committed probes from `stage2_chatbot/eval/probe_set.jsonl`.
Roughly 5–10 minutes per model on a T4, plus a one-off ~6 GB download each.

**Each candidate carries its own prompt format, and this is not cosmetic.** The first
bake-off ran MaLLaM through the plain `User:/Assistant:` fallback because its
tokenizer ships no `chat_template`, and every one of its 20 replies was degenerate —
it never emitted a turn-ending token and carried on inventing `User:` exchanges with
itself. That measured the fallback, not the model. MaLLaM's card states it uses the
exact Mistral Instruct template, so it is now forced with `--chat-format mistral`.

If MaLLaM wins, `stage2_chatbot/training/qlora_config.yaml` needs `template: mistral` for the same
reason. A wrong template does not raise; it trains on mis-delimited text.

**Re-running only MaLLaM is fine** — comment out the `llama` line. Decoding is greedy,
so Llama's existing `stage2_chatbot/eval/results/llama.json` is byte-identical to what a re-run would
produce, and skipping it saves a 6 GB download.

4-bit loading is used here for the same reason it is used in training: it is the
configuration the model will actually be judged in.


In [ ]:
# tag -> (model id, chat format). 'auto' uses the tokenizer's own template.
CANDIDATES = {
    'llama':  ('meta-llama/Llama-3.2-3B-Instruct', 'auto'),
    'mallam': ('mesolitica/mallam-3b-20k-instructions', 'mistral'),
    # 'qwen': ('Qwen/Qwen2.5-3B-Instruct', 'auto'),
}

for tag, (model_id, fmt) in CANDIDATES.items():
    print()
    print('=' * 70)
    print(f'{tag}: {model_id}  [{fmt}]')
    print('=' * 70, flush=True)
    !python stage2_chatbot/eval/generate.py --model {model_id} --tag {tag} --4bit --chat-format {fmt}


## Cell 4 — Objective refusal table

Fallback accuracy and over-refusal, exact and loose. Instant, no GPU.

Expect both to score badly here — neither has been taught the fallback line, so
`exact` will likely be 0% for both. That is not a broken script; it is the
measurement that makes the fine-tuned comparison at Step 4 mean something. What to
read at this stage is `loose`: whether a candidate declines at all when it should.


In [ ]:
!python stage2_chatbot/eval/refusal_report.py --runs {','.join(CANDIDATES)} --show-misses


## Cell 5 — Build the rating sheet and save to Drive

**Do not skip.** Everything under `/content` is lost on disconnect, and Colab
disconnects on idle. Drive is what makes these survive.

`rating_key.json` un-blinds the sheet — leave it unopened until both members have
finished rating.


In [ ]:
!python stage2_chatbot/eval/make_rating_sheet.py --runs {','.join(CANDIDATES)}

import shutil, pathlib
OUT = pathlib.Path('/content/drive/MyDrive/RDS3S1/NLP/bakeoff')
OUT.mkdir(parents=True, exist_ok=True)
SRC = pathlib.Path('/content/NLP-BorakBot/stage2_chatbot/eval')

for name in ['rating_sheet.csv', 'rating_key.json', 'probe_set.jsonl']:
    shutil.copy2(SRC / name, OUT / name)
shutil.copytree(SRC / 'results', OUT / 'results', dirs_exist_ok=True)

for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f'{p.stat().st_size:>9,}  {p.relative_to(OUT)}')


## Cell 6 — What happens next

1. Open `/content/drive/MyDrive/RDS3S1/NLP/bakeoff/` in Drive.
2. Copy `results/*.json` and `rating_sheet.csv` into the repo under `stage2_chatbot/eval/` and
   commit them. The generations are the evidence behind the choice; a table without
   them is an assertion.
3. Each member takes their own copy of the sheet, rates every row 1–5 on the three
   axes, and saves it as `stage2_chatbot/eval/rating_sheet_<initials>.csv`. Rate independently —
   comparing as you go destroys the agreement number.
4. Locally: `python stage2_chatbot/eval/score_ratings.py`
5. Write `docs/Archive/model_selection.md`: the table, the winner, one paragraph of why.
   If the spread is under ~0.3 the candidates are tied — pick on tooling risk and
   say so plainly.

Then Step 3: convert the splits to LLaMA-Factory format and train, also on Colab.
